# Notebook 87: Smoothed Exit Signals (Moving Average Filtering)

**Date**: 2026-01-25  
**Strategy**: Buy The Dip entries + Smoothed on-chain exits  
**Hypothesis**: Moving averages filter out noise from hourly spikes

---

## 🎯 Problem Statement

From Notebook 85 analysis, we found:
- **LTH-SOPR > 1.5 occurs 55% of hours** (too frequent!)
- **Brief spikes** average 5.8 hours (intraday noise)
- **Not sustained distribution** = false exit signals

## 💡 Solution: Moving Average Smoothing

Instead of:
```python
exit_signal = lth_sopr > 1.5  # Raw value (noisy)
```

Use:
```python
lth_sopr_ma7 = lth_sopr.rolling(7).mean()  # 7-day average
exit_signal = lth_sopr_ma7 > 1.5  # Smoothed (filters spikes)
```

**Benefit**: Only triggers when metric is **sustained** above threshold for days, not hours!

---

## 📊 Test Matrix

We'll test combinations of:

### Moving Average Windows
- **MA-3**: 3-day average (responsive)
- **MA-7**: 7-day average (balanced)
- **MA-14**: 14-day average (smooth)
- **MA-21**: 21-day average (very smooth)

### Metrics
- **LTH-SOPR**: Long-term holder profit taking
- **STH-SOPR**: Short-term holder sentiment
- **MVRV**: Market value vs realized value

### Thresholds
- **LTH-SOPR**: 1.3, 1.5, 1.8, 2.0
- **STH-SOPR**: 1.1, 1.2, 1.3
- **MVRV**: 2.0, 2.5, 3.0

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

DATA_DIR = Path('../data')
BRK_DAILY = DATA_DIR / 'brk' / 'daily'
GN_DAILY = DATA_DIR / 'glassnode' / 'daily'

START_DATE = '2020-02-02'
TRANSACTION_COST = 0.001

print("✅ Setup complete")
print(f"📅 Backtest period: {START_DATE} → present")
print(f"💸 Transaction cost: {TRANSACTION_COST * 100:.1f}%")

---

## 1. Load Data

In [ ]:
print("Loading daily data...")

# Price & on-chain metrics
price_daily = pd.read_parquet(BRK_DAILY / 'price.parquet')
mvrv_daily = pd.read_parquet(BRK_DAILY / 'mvrv.parquet')
mvrv_sth_daily = pd.read_parquet(BRK_DAILY / 'mvrv_sth.parquet')
sopr_sth_daily = pd.read_parquet(BRK_DAILY / 'sopr_sth.parquet')
sopr_lth_daily = pd.read_parquet(BRK_DAILY / 'sopr_lth.parquet')
realized_profit_daily = pd.read_parquet(BRK_DAILY / 'realized_profit.parquet')
realized_loss_daily = pd.read_parquet(BRK_DAILY / 'realized_loss.parquet')

# Derivatives
funding_rate_daily = pd.read_parquet(GN_DAILY / 'funding_rate.parquet')
liq_long_daily = pd.read_parquet(GN_DAILY / 'liquidations_long.parquet')
liq_short_daily = pd.read_parquet(GN_DAILY / 'liquidations_short.parquet')

# Normalize
for df in [price_daily, mvrv_daily, mvrv_sth_daily, sopr_sth_daily, sopr_lth_daily,
           realized_profit_daily, realized_loss_daily,
           funding_rate_daily, liq_long_daily, liq_short_daily]:
    if 'time' not in df.columns:
        df.reset_index(inplace=True)
    df['time'] = pd.to_datetime(df['time'])
    df.sort_values('time', inplace=True)

# Filter to backtest period
price_daily = price_daily[price_daily['time'] >= START_DATE].copy()
mvrv_daily = mvrv_daily[mvrv_daily['time'] >= START_DATE].copy()
mvrv_sth_daily = mvrv_sth_daily[mvrv_sth_daily['time'] >= START_DATE].copy()
sopr_sth_daily = sopr_sth_daily[sopr_sth_daily['time'] >= START_DATE].copy()
sopr_lth_daily = sopr_lth_daily[sopr_lth_daily['time'] >= START_DATE].copy()
realized_profit_daily = realized_profit_daily[realized_profit_daily['time'] >= START_DATE].copy()
realized_loss_daily = realized_loss_daily[realized_loss_daily['time'] >= START_DATE].copy()
funding_rate_daily = funding_rate_daily[funding_rate_daily['time'] >= START_DATE].copy()
liq_long_daily = liq_long_daily[liq_long_daily['time'] >= START_DATE].copy()
liq_short_daily = liq_short_daily[liq_short_daily['time'] >= START_DATE].copy()

print(f"✅ Loaded {len(price_daily):,} daily bars")
print(f"   Range: {price_daily['time'].min().date()} → {price_daily['time'].max().date()}")

---

## 2. Build Entry Signals (Buy The Dip)

In [ ]:
# Merge daily data
daily_data = price_daily[['time', 'value']].rename(columns={'value': 'price'})
daily_data = daily_data.merge(mvrv_daily[['time', 'value']].rename(columns={'value': 'mvrv'}), on='time', how='left')
daily_data = daily_data.merge(mvrv_sth_daily[['time', 'value']].rename(columns={'value': 'mvrv_sth'}), on='time', how='left')
daily_data = daily_data.merge(sopr_sth_daily[['time', 'value']].rename(columns={'value': 'sopr_sth'}), on='time', how='left')
daily_data = daily_data.merge(sopr_lth_daily[['time', 'value']].rename(columns={'value': 'sopr_lth'}), on='time', how='left')
daily_data = daily_data.merge(realized_profit_daily[['time', 'value']].rename(columns={'value': 'realized_profit'}), on='time', how='left')
daily_data = daily_data.merge(realized_loss_daily[['time', 'value']].rename(columns={'value': 'realized_loss'}), on='time', how='left')
daily_data = daily_data.merge(funding_rate_daily[['time', 'value']].rename(columns={'value': 'funding_rate'}), on='time', how='left')
daily_data = daily_data.merge(liq_long_daily[['time', 'value']].rename(columns={'value': 'liq_long'}), on='time', how='left')
daily_data = daily_data.merge(liq_short_daily[['time', 'value']].rename(columns={'value': 'liq_short'}), on='time', how='left')

# Calculate derived metrics
daily_data['rpl_ratio'] = daily_data['realized_profit'] / daily_data['realized_loss']
daily_data['liq_ratio'] = daily_data['liq_long'] / daily_data['liq_short']

# Calculate moving averages for exit metrics
for window in [3, 7, 14, 21]:
    daily_data[f'lth_sopr_ma{window}'] = daily_data['sopr_lth'].rolling(window=window, min_periods=1).mean()
    daily_data[f'sth_sopr_ma{window}'] = daily_data['sopr_sth'].rolling(window=window, min_periods=1).mean()
    daily_data[f'mvrv_ma{window}'] = daily_data['mvrv'].rolling(window=window, min_periods=1).mean()

# Buy The Dip entry conditions
daily_data['cond1_sth_mvrv'] = daily_data['mvrv_sth'] < 1.0
daily_data['cond2_sth_sopr'] = daily_data['sopr_sth'] < 1.0
daily_data['cond3_rpl_ratio'] = daily_data['rpl_ratio'] < 1.0
daily_data['cond4_funding'] = daily_data['funding_rate'] <= 0.0
daily_data['cond5_liquidations'] = daily_data['liq_ratio'] > 1.0

daily_data['conditions_met'] = (
    daily_data['cond1_sth_mvrv'].astype(int) +
    daily_data['cond2_sth_sopr'].astype(int) +
    daily_data['cond3_rpl_ratio'].astype(int) +
    daily_data['cond4_funding'].astype(int) +
    daily_data['cond5_liquidations'].astype(int)
)

daily_data['entry_signal'] = daily_data['conditions_met'] >= 4

print(f"✅ Signals calculated")
print(f"   Entry signals: {daily_data['entry_signal'].sum()} days")
print(f"   Moving averages calculated: MA-3, MA-7, MA-14, MA-21")

---

## 3. Backtest Engine with Smoothed Exits

In [ ]:
def backtest_smoothed_exits(df, exit_metric, ma_window, threshold):
    """
    Backtest with smoothed exit signals.
    
    Parameters:
    -----------
    df : DataFrame with daily data + signals
    exit_metric : Column name for exit metric (e.g., 'lth_sopr_ma7')
    ma_window : MA window for display (e.g., 7)
    threshold : Exit threshold value
    
    Returns:
    --------
    trades : List of completed trades
    metrics : Performance metrics dict
    """
    
    position = None
    capital = 10_000
    trades = []
    
    for idx, row in df.iterrows():
        date = row['time']
        price = row['price']
        
        # Entry
        if position is None and row['entry_signal']:
            capital_after_fee = capital * (1 - TRANSACTION_COST)
            size = capital_after_fee / price
            
            position = {
                'entry_date': date,
                'entry_price': price,
                'size': size
            }
            capital = 0
        
        # Exit check
        if position is not None:
            hold_days = (date - position['entry_date']).days
            
            # Smoothed exit signal
            exit_triggered = row[exit_metric] > threshold if exit_metric in row and not pd.isna(row[exit_metric]) else False
            
            # Also add max hold (60 days) and stop loss (-10%)
            pnl_pct = (price / position['entry_price']) - 1
            stop_loss = pnl_pct <= -0.10
            max_hold = hold_days >= 60
            
            if exit_triggered or stop_loss or max_hold:
                exit_value = position['size'] * price * (1 - TRANSACTION_COST)
                
                exit_type = 'SMOOTHED_EXIT' if exit_triggered else ('STOP' if stop_loss else 'MAX_HOLD')
                
                trades.append({
                    'entry_date': position['entry_date'],
                    'exit_date': date,
                    'entry_price': position['entry_price'],
                    'exit_price': price,
                    'hold_days': hold_days,
                    'pnl_pct': (exit_value / 10_000 - 1) * 100,
                    'exit_type': exit_type
                })
                
                capital = exit_value
                position = None
    
    # Close final position if still open
    if position is not None:
        final_price = df['price'].iloc[-1]
        exit_value = position['size'] * final_price * (1 - TRANSACTION_COST)
        capital = exit_value
    
    # Calculate metrics
    if len(trades) == 0:
        return [], None
    
    trades_df = pd.DataFrame(trades)
    
    total_return = (capital / 10_000 - 1) * 100
    num_trades = len(trades_df)
    winners = (trades_df['pnl_pct'] > 0).sum()
    win_rate = winners / num_trades * 100
    avg_return = trades_df['pnl_pct'].mean()
    avg_hold = trades_df['hold_days'].mean()
    
    years = (df['time'].max() - df['time'].min()).days / 365.25
    trades_per_year = num_trades / years
    
    metrics = {
        'total_return': total_return,
        'num_trades': num_trades,
        'trades_per_year': trades_per_year,
        'win_rate': win_rate,
        'avg_return': avg_return,
        'avg_hold': avg_hold,
        'ma_window': ma_window,
        'threshold': threshold
    }
    
    return trades, metrics

print("✅ Backtest engine ready")

---

## 4. Test LTH-SOPR with Different MA Windows

In [ ]:
print("="*80)
print("TESTING: LTH-SOPR with Moving Average Smoothing")
print("="*80)

# Test matrix
ma_windows = [3, 7, 14, 21]
thresholds = [1.3, 1.5, 1.8, 2.0]

results = []

for ma_window in ma_windows:
    for threshold in thresholds:
        metric_col = f'lth_sopr_ma{ma_window}'
        trades, metrics = backtest_smoothed_exits(
            daily_data,
            exit_metric=metric_col,
            ma_window=ma_window,
            threshold=threshold
        )
        
        if metrics:
            results.append(metrics)
            print(f"MA-{ma_window:>2} @ {threshold:>3.1f}: Return={metrics['total_return']:>8.1f}%  "
                  f"Trades={metrics['num_trades']:>3}  WinRate={metrics['win_rate']:>5.1f}%  "
                  f"Avg={metrics['avg_return']:>6.2f}%")

print(f"\n✅ Tested {len(results)} combinations")

---

## 5. Find Best Configuration

In [ ]:
if len(results) > 0:
    results_df = pd.DataFrame(results)
    
    # Buy & Hold for comparison
    bh_return = (daily_data['price'].iloc[-1] / daily_data['price'].iloc[0] - 1) * 100
    
    print("\n" + "="*80)
    print("TOP 10 CONFIGURATIONS (by Total Return)")
    print("="*80)
    
    top_10 = results_df.nlargest(10, 'total_return')
    
    for idx, row in top_10.iterrows():
        alpha = row['total_return'] - bh_return
        print(f"\n#{idx+1} MA-{row['ma_window']:.0f} @ LTH-SOPR > {row['threshold']:.1f}")
        print(f"   Return:        {row['total_return']:>8.1f}%  (vs B&H {bh_return:.0f}%: {alpha:+.0f}%)")
        print(f"   Trades:        {row['num_trades']:>8.0f} ({row['trades_per_year']:.1f}/year)")
        print(f"   Win Rate:      {row['win_rate']:>8.1f}%")
        print(f"   Avg Return:    {row['avg_return']:>8.2f}%")
        print(f"   Avg Hold:      {row['avg_hold']:>8.1f} days")
    
    # Best by different criteria
    print("\n" + "="*80)
    print("BEST BY CRITERIA")
    print("="*80)
    
    best_return = results_df.loc[results_df['total_return'].idxmax()]
    print(f"\n🏆 Best Return: MA-{best_return['ma_window']:.0f} @ {best_return['threshold']:.1f}")
    print(f"   Return: {best_return['total_return']:.1f}%")
    
    best_winrate = results_df.loc[results_df['win_rate'].idxmax()]
    print(f"\n🎯 Best Win Rate: MA-{best_winrate['ma_window']:.0f} @ {best_winrate['threshold']:.1f}")
    print(f"   Win Rate: {best_winrate['win_rate']:.1f}%")
    
    best_frequency = results_df.loc[results_df['trades_per_year'].idxmax()]
    print(f"\n🔄 Most Frequent: MA-{best_frequency['ma_window']:.0f} @ {best_frequency['threshold']:.1f}")
    print(f"   Trades/Year: {best_frequency['trades_per_year']:.1f}")
    
else:
    print("\n⚠️  No valid results")

---

## 6. Compare: Raw vs Smoothed

In [ ]:
print("\n" + "="*80)
print("RAW vs SMOOTHED COMPARISON")
print("="*80)

# Test raw LTH-SOPR (no smoothing)
print("\n📊 RAW LTH-SOPR (no moving average):")

for threshold in [1.3, 1.5, 1.8, 2.0]:
    trades, metrics = backtest_smoothed_exits(
        daily_data,
        exit_metric='sopr_lth',
        ma_window=0,
        threshold=threshold
    )
    
    if metrics:
        print(f"  Threshold {threshold:>3.1f}: Return={metrics['total_return']:>8.1f}%  "
              f"Trades={metrics['num_trades']:>3}  WinRate={metrics['win_rate']:>5.1f}%")

print("\n📊 SMOOTHED LTH-SOPR (MA-7 - balanced):")

for threshold in [1.3, 1.5, 1.8, 2.0]:
    trades, metrics = backtest_smoothed_exits(
        daily_data,
        exit_metric='lth_sopr_ma7',
        ma_window=7,
        threshold=threshold
    )
    
    if metrics:
        print(f"  Threshold {threshold:>3.1f}: Return={metrics['total_return']:>8.1f}%  "
              f"Trades={metrics['num_trades']:>3}  WinRate={metrics['win_rate']:>5.1f}%")

print("\n💡 Analysis:")
print("   If smoothed > raw: MA filtering works! (less noise)")
print("   If raw > smoothed: MA delays exits too much (miss optimal exits)")

---

## 7. Test Other Metrics (MVRV, STH-SOPR)

In [ ]:
print("\n" + "="*80)
print("TESTING: MVRV with Smoothing")
print("="*80)

for ma_window in [7, 14, 21]:
    for threshold in [2.0, 2.5, 3.0]:
        metric_col = f'mvrv_ma{ma_window}'
        trades, metrics = backtest_smoothed_exits(
            daily_data,
            exit_metric=metric_col,
            ma_window=ma_window,
            threshold=threshold
        )
        
        if metrics:
            print(f"MA-{ma_window:>2} @ {threshold:>3.1f}: Return={metrics['total_return']:>8.1f}%  "
                  f"Trades={metrics['num_trades']:>3}  WinRate={metrics['win_rate']:>5.1f}%")

print("\n" + "="*80)
print("TESTING: STH-SOPR with Smoothing")
print("="*80)

for ma_window in [7, 14, 21]:
    for threshold in [1.1, 1.2, 1.3]:
        metric_col = f'sth_sopr_ma{ma_window}'
        trades, metrics = backtest_smoothed_exits(
            daily_data,
            exit_metric=metric_col,
            ma_window=ma_window,
            threshold=threshold
        )
        
        if metrics:
            print(f"MA-{ma_window:>2} @ {threshold:>3.1f}: Return={metrics['total_return']:>8.1f}%  "
                  f"Trades={metrics['num_trades']:>3}  WinRate={metrics['win_rate']:>5.1f}%")

---

## 8. Summary & Conclusions

In [ ]:
print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)

print("\n💡 Key Findings:")
print("\n1. Does Moving Average Smoothing Work?")
print("   → Check if smoothed returns > raw returns")
print("   → If yes: MA filtering removes noise successfully")
print("   → If no: Delays exit too much, miss optimal timing")

print("\n2. Optimal MA Window:")
print("   → MA-3: More responsive, catches early exits")
print("   → MA-7: Balanced (filter noise, stay responsive)")
print("   → MA-14/21: Very smooth, but may delay exits")

print("\n3. Threshold Selection:")
print("   → Lower (1.3-1.5): More exits, more frequent")
print("   → Higher (1.8-2.0): Fewer exits, only strong signals")

print("\n4. Practical Application:")
print("   If smoothing helps:")
print("     • Use MA-7 LTH-SOPR > X as exit condition")
print("     • Requires metric sustained above threshold for ~1 week")
print("     • Filters out brief intraday spikes")
print("\n   If smoothing hurts:")
print("     • Stick with daily raw values")
print("     • Use higher thresholds instead (2.5, 3.0)")
print("     • Or use different exit logic (price-based, momentum)")

print("\n✅ Analysis complete")